In [3]:

import requests
import os

# NYC TLC source URL
base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/"

# Multiple years and months
years = ["2025", "2026"]
months = ["01", "02", "03", "04"]

# Temporary folder
local_folder = "/tmp/nyc_taxi"
os.makedirs(local_folder, exist_ok=True)

# Loop through years
for year in years:

    # Loop through months
    for month in months:

        filename = f"yellow_tripdata_{year}-{month}.parquet"
        url = base_url + filename
        local_path = os.path.join(local_folder, filename)

        print(f"Downloading {filename}")

        try:
            with requests.get(url, stream=True, timeout=120) as response:
                response.raise_for_status()

                with open(local_path, "wb") as f:
                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):
                        if chunk:
                            f.write(chunk)

            print(f"Downloaded successfully: {filename}")

        except requests.exceptions.HTTPError as e:
            print(f"HTTP error for {filename}: {e}")

        except requests.exceptions.RequestException as e:
            print(f"Download failed for {filename}: {e}")

print("Download process completed")


StatementMeta(Customer360, 2, 4, Finished, Available, Finished, False)

Downloaded successfully: yellow_tripdata_2025-01.parquet
Downloaded successfully: yellow_tripdata_2025-02.parquet
Downloaded successfully: yellow_tripdata_2025-03.parquet
Downloaded successfully: yellow_tripdata_2025-04.parquet
Downloaded successfully: yellow_tripdata_2026-01.parquet
Downloaded successfully: yellow_tripdata_2026-02.parquet
Downloaded successfully: yellow_tripdata_2026-03.parquet
Downloaded successfully: yellow_tripdata_2026-04.parquet
Download process completed


In [5]:

from notebookutils import mssparkutils
import os

# ADLS configuration
storage_account = "adls360demo01"
container = "customer360"

# Years and months
years = ["2025", "2026"]
months = ["01", "02", "03", "04"]

# Local folder containing downloaded Parquet files
local_folder = "/tmp/nyc_taxi"

# ADLS destination
adls_folder = (
    f"abfss://{container}@{storage_account}.dfs.core.windows.net/"
    "Taxi_yellow_page/raw/"
)

# Create ADLS folder if it does not exist
mssparkutils.fs.mkdirs(adls_folder)

# Loop through years
for year in years:

    # Loop through months
    for month in months:

        filename = f"yellow_tripdata_{year}-{month}.parquet"

        source_path = f"{local_folder}/{filename}"
        destination = adls_folder + filename

        # Check whether file exists locally
        if os.path.exists(source_path):

            # Copy file to ADLS
            mssparkutils.fs.cp(
                f"file:{source_path}",
                destination
            )

            print(f"Uploaded to ADLS: {filename}")

        else:
            print(f"File not found, skipping: {filename}")

print("ADLS upload process completed")



StatementMeta(Customer360, 2, 6, Finished, Available, Finished, False)

Uploaded to ADLS: yellow_tripdata_2025-01.parquet
Uploaded to ADLS: yellow_tripdata_2025-02.parquet
Uploaded to ADLS: yellow_tripdata_2025-03.parquet
Uploaded to ADLS: yellow_tripdata_2025-04.parquet
Uploaded to ADLS: yellow_tripdata_2026-01.parquet
Uploaded to ADLS: yellow_tripdata_2026-02.parquet
Uploaded to ADLS: yellow_tripdata_2026-03.parquet
Uploaded to ADLS: yellow_tripdata_2026-04.parquet
ADLS upload process completed


In [8]:

from pyspark.sql.functions import *


source_path = (
    "abfss://customer360@adls360demo01.dfs.core.windows.net/"
    "Taxi_yellow_page/raw/"
)

# Read Parquet files from both year folders
df = spark.read.option("recursiveFileLookup", "true").parquet(source_path)

# Display data
display(df)

# Check schema
df.printSchema()

# Count records
print("Total records:", df.count())


StatementMeta(Customer360, 2, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 929f0269-a202-4b32-91f3-6e1ed330d351)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

Total records: 30077025


In [9]:

df_transformed = (
    df.withColumn("year", year(col("tpep_pickup_datetime")))
      .withColumn("month", month(col("tpep_pickup_datetime")))
)

target_path = (
    "abfss://customer360@adls360demo01.dfs.core.windows.net/"
    "Taxi_yellow_delta/"
)

df_transformed.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .save(target_path)

print("Delta conversion completed successfully")


StatementMeta(Customer360, 2, 10, Finished, Available, Finished, False)

Delta conversion completed successfully


In [10]:

delta_df = spark.read.format("delta").load(target_path)

display(
    delta_df.groupBy("year", "month")
    .count()
    .orderBy("year", "month")
)


StatementMeta(Customer360, 2, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f5c32722-6d2a-4c42-81f2-bc3857e7f299)